In [1]:
import os
os.environ["OMP_NUM_THREADS"] = '1'

In [2]:
import numpy as np
import math
import matplotlib.pyplot as plt
import random
from scipy.stats import poisson, norm
from scipy.optimize import least_squares
from sklearn.cluster import KMeans
import time



from StockPathSimulation import StockPathSimulation
from StrategySimulation import StrategySimulation

### Parameter Optimization on Simulated Data

In [4]:
t = 50/252
nSims = 10
nSteps = 50

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])

#### Nonlinear Least Squares

We will use the method of nonlinear least squares to find the optimal parameters for three different models. We set somewhat arbitrary but consistent bounds for the parameters and use a random value (within the bounds) as the initial guess for the parameters.

First, we consider the price of a call option when the underlying stock is modeled by geometric Brownian motion.

In [7]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):

    model_prices = ss.kappa(stockPrices,strikePrices, guess)

    return (marketPrices - model_prices).flatten()

In [8]:
lowerBounds = np.array([
    .05  # lower bound for volatility
])

upperBounds = np.array([
    .9  # upper bound for volatility
])



In [9]:
for i in range(10):

    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(interestRate, 0.4)
    
    process = sps.simGeomBrownianMotion(meanRateOfReturn = trueMeanRateOfReturn,
                                       volatility = trueVolatility,
                                       initialPrice = initialPrice)
    
    optionPrice = ss.kappa(process, strikePrices, trueVolatility)
    
    guess = random.uniform(lowerBounds[0], upperBounds[0])
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices,process,optionPrice))
    
    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print()


Results for Trial 1

Message: `gtol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.24126062505661658 and the true volatility is 0.2412606250566166
The value of the cost function at the estimated parameter is 2.9505265296328454e-26
Are there any parameters that are at the bounds?: False

Results for Trial 2

Message: `gtol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.7714852669345832 and the true volatility is 0.7714852669345832
The value of the cost function at the estimated parameter is 0.0
Are there any parameters that are at the bounds?: False

Results for Trial 3

Message: `gtol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.32088663656169586 and the true volatility is 0.3208866365616863
The value of the cost function at the estimated parameter is 7.584322464444488e-24
Are there any parameters that are at the bounds?: False

Results for Trial 4

Message: `gto

Next, we consider the price of a call option when the underlying stock is modeled by
\begin{equation*}
S(t) = S(0) e^{(\alpha - \lambda \sigma)t} (\sigma+1)^{N(t)},
\end{equation*}
where N(t) is a Poisson process with intensity $\lambda > 0$.

In [11]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility, ltild = guess
    model_prices = ss.poissonCallPrice(stockPrices, strikePrices, volatility, ltild, error)

    return (marketPrices - model_prices).flatten()

In [12]:
error = 1e-6

lowerBounds = np.array([
    .05,  # lower bound for volatility
    .9    # lower bound for parameter 
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    31   # upper bound for parameter
])


for i in range(10):

    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(.005, .1)
    trueIntensity = random.uniform(1.0, 30.0)
    trueParam = trueIntensity - ((trueMeanRateOfReturn - ss.r)/trueVolatility)
    
    
    process = sps.assetModel1(meanRateOfReturn = trueMeanRateOfReturn,
                              volatility = trueVolatility,
                              intensity = trueIntensity,
                              initialPrice = initialPrice)
    
    optionPrice = ss.poissonCallPrice(process, strikePrices, trueVolatility, trueParam, error)
    
    guess = [
        random.uniform(lowerBounds[0], upperBounds[0]),
        random.uniform(lowerBounds[1], upperBounds[1])
    ]
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice))
    
    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The estimated parameter is {result['x'][1]} and the true parameter is {trueParam}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print()

Results for Trial 1

Message: `ftol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.08727128176996653 and the true volatility is 0.10317176421967802
The estimated parameter is 30.013266440166284 and the true parameter is 21.246270485969692
The value of the cost function at the estimated parameter is 5.965667824322992
Are there any parameters that are at the bounds?: False

Results for Trial 2

Message: `ftol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.09716232641733644 and the true volatility is 0.07281888163517447
The estimated parameter is 4.453986241144021 and the true parameter is 7.091511538536494
The value of the cost function at the estimated parameter is 3.7185918271699103
Are there any parameters that are at the bounds?: False

Results for Trial 3

Message: `ftol` termination condition is satisfied.
Success status: True
The estimated volatility is 0.3010554938657975 and the true volatility is 0

Finally, we consider the price of a call option when the underlying stock is modeled by
\begin{equation*}
S(t)=S(0)\text{exp}\biggl\{\sigma W(t) + \left( \alpha - \beta\lambda -\frac{1}{2}\sigma^2\right) t \biggr\} \prod_{i=1}^{N(t)}(Y_i+1).
\end{equation*}
where $N(t)$ is a Poisson process with intensity $\lambda$ and $Y_1, Y_2, \dotsc$ form a sequence of independent and identically distributed random variables with mean $\beta = \mathbb{E}(Y_i)$. We further assume that, for each $i$, $Y_i$ takes on finitely many values $y_1, \dotsc, y_M$ and write $p(y_m) = P(\{Y_i = y_m\})$.

We will consider the cases when $M = 2$ and $M = 3$.

In [14]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    numParams = (len(guess) - 1) // 2
    volatility = guess[0]
    jumps = guess[1:1 + numParams]
    params = guess[1+numParams:]
    model_prices = ss.jumpDiffusionCallPrice(stockPrices, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

In [15]:
error = 1e-6

lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    0,    # lower bound for parameters
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    10,  # upper bound for parameters
    10
])


for i in range(10):

    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(.005, .3)
    trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 2)
    trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
    while (np.any(trueParameters) == False):
        trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
    
    
    
    
    process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                              volatility = trueVolatility,
                              intensity = np.sum(trueParameters),
                              jumps = trueJumps,
                              jumpProbabilities = trueParameters/np.sum(trueParameters),
                              initialPrice = initialPrice)
    
    optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)

    guess = np.random.uniform(low = lowerBounds, high = upperBounds)

    start = time.perf_counter()
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice), max_nfev=80)
    
    end = time.perf_counter()

    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The inital guesses are {guess}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The estimated jump sizes are {result['x'][1:3]} and the true jump sizes are {trueJumps}")
    print(f"The estimated parameters are {result['x'][3:]} and the true parameters are {trueParameters}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print(f'Time for least squares optimization: {end - start} seconds') 
    print()

Results for Trial 1

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [0.14448951 0.46832478 0.73495094 1.08746744 6.22687567]
The estimated volatility is 0.08749453363444716 and the true volatility is 0.08749453363441684
The estimated jump sizes are [0.95322655 0.79496155] and the true jump sizes are [0.95322655 0.79496155]
The estimated parameters are [6.84068962 6.37284532] and the true parameters are [6.84068962 6.37284532]
The value of the cost function at the estimated parameter is 1.382602949918063e-23
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 24.12605420005275 seconds

Results for Trial 2

Message: The maximum number of function evaluations is exceeded.
Success status: False
The inital guesses are [0.77911575 0.65318511 0.4341938  0.46349334 7.7667551 ]
The estimated volatility is 0.32488071019715 and the true volatility is 0.32449489840830226
The estimated jump sizes are [-0.02780

In [16]:
lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    -0.6,
    0,    # lower bound for parameters
    0,
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    1,
    10,  # upper bound for parameters
    10,
    10
])


for i in range(10):

    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(.005, .3)
    trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 3)
    trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
    while (np.any(trueParameters) == False):
        trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
    
    
    
    
    process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                              volatility = trueVolatility,
                              intensity = np.sum(trueParameters),
                              jumps = trueJumps,
                              jumpProbabilities = trueParameters/np.sum(trueParameters),
                              initialPrice = initialPrice)
    
    optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)
    
    
    guess = np.random.uniform(low = lowerBounds, high = upperBounds)
    
    start = time.perf_counter()
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, process, optionPrice), max_nfev=80)
    
    end = time.perf_counter()
    
    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The inital guesses are {guess}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The estimated jump sizes are {result['x'][1:3]} and the true jump sizes are {trueJumps}")
    print(f"The estimated parameters are {result['x'][3:]} and the true parameters are {trueParameters}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print(f'Time for least squares optimization: {end - start} seconds') 
    print()

Results for Trial 1

Message: The maximum number of function evaluations is exceeded.
Success status: False
The inital guesses are [ 0.8340571  -0.0879002   0.56638154 -0.04830591  5.61663541  4.09394247
  6.35161232]
The estimated volatility is 0.2303235536408844 and the true volatility is 0.23119539456358368
The estimated jump sizes are [-0.59310881  0.46921663] and the true jump sizes are [ 0.4692172  -0.59310881 -0.02206817]
The estimated parameters are [-0.01581064  8.98924292  8.65566712  5.34279821] and the true parameters are [8.65563134 8.98924062 1.92524395]
The value of the cost function at the estimated parameter is 2.6937173695051815e-08
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 618.9478979999549 seconds

Results for Trial 2

Message: The maximum number of function evaluations is exceeded.
Success status: False
The inital guesses are [ 2.99233504e-01  8.10029994e-01 -2.50954386e-01 -8.91683289e-03
  3.96330070e+00  9.75408

#### Nonlinear Least Squares with "Good Guesses"

As we can see, the least squares optimization takes a long time to converge even when $M$ is relatively low. To fix this issue, we attempt to make "good guesses" of where the optimal parameters should be before we perform least squares optimization. To do this, we use the ideas from "Econometrics of Testing for Jumps in Financial Economics Using Bipower Variation" by Barndoff-Nielsen and Shephard.

Let
\begin{equation*}
Y(t) = Y(0) + \sigma W(t) + \left ( \alpha - \beta \lambda - \frac{1}{2} \sigma^2 \right )t + \sum_{i=1}^{N(t)} \log(Y_i + 1)
\end{equation*}
denote the log price of the stock $S(t)$. Let $n > 0$ be a positive integer. Then, for any $t > 0$,
\begin{equation*}
a_n = \sum_{j=1}^n \left |Y\left ( \frac{jt}{n}\right ) - Y\left ( \frac{(j-1)t}{n} \right ) \right |^2
\end{equation*}
is a consistent estimator of $\sigma^2 t + \sum_{i=1}^{N(t)} \log(Y_i + 1)^2$ and 
\begin{equation*}
b_n = \frac{\pi}{2} \sum_{j=2}^n \left |Y\left ( \frac{jt}{n}\right ) - Y\left ( \frac{(j-1)t}{n} \right ) \right | \left |Y\left ( \frac{(j-1)t}{n}\right ) - Y\left ( \frac{(j-2)t}{n} \right ) \right |,
\end{equation*}
is a consistent estimator of $\sigma^2 t$ as $n \to \infty$. It follows that, $a_n - b_n$ is a consistent estimator of $\sum_{i=1}^{N(t)} \log(Y_i + 1)^2$ as $n \to \infty$. Moreover, under the null hypothesis that $N(t)$ is identically zero for all $t$,
\begin{equation*}
\frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} } \frac{\sqrt{n}(a_n - b_n)}{\sqrt{t}\sigma^2}
\end{equation*}
converges to the standard normal distribution as $n \to \infty$. We will use these facts to make our "good guesses".

Let $\{y_0, y_1, \dotsc, y_{nN}\}$ denote the observed values for $Y\left ( 0 \right ), Y\left ( \frac{1}{252*n} \right ), \dotsc, Y\left ( \frac{N}{252} \right )$. We calculate the sample bi-power variations for each day using the formula
\begin{equation*}
b_{nN,d} = \frac{\pi}{2} \sum\limits_{i=n*d+1}^{n*(d+1)-1} |y_{i+1} - y_i||y_i - y_{i-1}|
\end{equation*}
for each $d = 0, \dotsc, N-1$.

In [45]:
def sampleBiPowerVariation(stock, n):
    logRet = np.log(stock[:,1:]) - np.log(stock[:,:-1])
    
    nRows, nCols = logRet.shape
    dailyLogRet = logRet.reshape(nRows, -1, n)

    res  = 1.570796 * np.abs(dailyLogRet[:,:,1:]) * np.abs(dailyLogRet[:,:,:-1])
    return res.sum(axis=2)

For each $d = 0, \dotsc, N-1$, set
\begin{equation*}
a_{nN,d} = \sum_{i=n*d+1}^{n*(d+1)} |y_i - y_{i-1}|^2.
\end{equation*}
For each $d$, we calculate
\begin{equation*}
\hat{g}_d = \frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }\frac{\sqrt{n}}{\sqrt{252}} \frac{a_{nN,d} - b_{nN,d}}{b_{nN,d}}
\end{equation*}
We set the jump days to be
\begin{equation*}
\{ d = 0, \dotsc, N-1 : \hat{g}_d > 2.326\}.
\end{equation*}



Set $n = 390$ (so that we are considering minute-by-minute data). On the jump days, we expect
\begin{equation*}
\begin{split}
\hat{g}_d & \approx \frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }\frac{\sqrt{n}}{\sqrt{t}} \frac{\sum_{i=1}^{N(t)} \log(Y_i + 1)^2}{\sigma^2} \\
    & = \frac{\sqrt{390*252}}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }  \frac{\sum_{i=1}^{N(1/252)} \log(Y_i + 1)^2}{\sigma^2} \\
    & \approx 400 * \frac{\sum_{i=1}^{N(1/252)} \log(Y_i + 1)^2}{\sigma^2}
\end{split}
\end{equation*}
If, for all $i = n*d + 1, \dotsc, n*(d+1)$,
\begin{equation*}
|y_{i} - y_{i-1}|^2 - b_{nN,d}/n \leq 90 (b_{nN,d} / n),
\end{equation*}
then
\begin{equation*}
\begin{split}
\hat{g}_d & = \frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }\frac{\sqrt{n}}{\sqrt{252}} \sum_{i=n*d+1}^{n*(d+1)} \frac{ |y_i - y_{i-1}|^2 - (b_{nN,d}/n)}{b_{nN,d}} \\
    & \leq \frac{1}{\sqrt{\frac{\pi^2}{4} + \pi - 5} }\frac{\sqrt{n}}{\sqrt{252}} * 90 \\
    & \approx 140.
\end{split}
\end{equation*}
So, on each jump day, we will view a change in price $|y_{i} - y_{i-1}|$ as a jump if
\begin{equation*}
|y_{i} - y_{i-1}|^2 - b_{nN,d}/n > 90 (b_{nN,d} / n).
\end{equation*}

In [48]:
def jumps(stock, n):
    logRet = np.log(stock[:,1:]) - np.log(stock[:,:-1])
    nRows, nCols = logRet.shape
    dailyLogRet = logRet.reshape(nRows, -1, n)
    
    b  = 1.570796 * np.abs(dailyLogRet[:,:,1:]) * np.abs(dailyLogRet[:,:,:-1])
    b = b.sum(axis=2)
    
    a = np.sum(dailyLogRet**2, axis=2)

    statistic = 1.281426 * np.sqrt(n/252) * (a-b)/b
    jumpMask = statistic > 2.326
    jumpMask = np.repeat(jumpMask[:,:,np.newaxis], n, axis = 2)

    b = np.repeat(b[:,:,np.newaxis], n, axis = 2)
    maxMask = (dailyLogRet**2 - b/n)/(b/n) > 90 

    mask = np.logical_and(maxMask, jumpMask)
    return dailyLogRet[mask]




Finally, we perform a clustering analysis on the observed jumps to determine the jump sizes and the jump probabilities of the stock path.

In [51]:
def jumpCluster(stock, n, M):
    js = jumps(stock, n)

    if len(js) < M:
        return np.ones(M) * np.mean(js), np.ones(M)/M
    
    js = js.reshape(-1, 1)
    
    kmeans = KMeans(n_clusters = M, n_init = 'auto')
    labels = kmeans.fit_predict(js)
    
    sizes = kmeans.cluster_centers_.flatten()
    probs = np.array([(labels == k).mean() for k in range(M)])
    
    order = np.argsort(sizes)
    sizes = sizes[order]
    probs = probs[order]
    
    return np.exp(sizes)-1, probs

Now, we use the functions we defined above to create our guesses when performing parameter optimization. We will look at call option prices and the prices of the underlying stocks 110 days before expiry. We will look at the stock prices of the first 60 days to create our initial guess for parameter optimization. Then, we will optimize our parameters on the last 50 days of call option prices and underlying stock prices.

In [54]:
t = 110/252
nSims = 10
nSteps = 390 * 110

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

t = 50/252
nSims = 10
nSteps = 50


interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])


In [56]:
lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    0,    # lower bound for parameters
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    10,  # upper bound for parameters
    10
])



for i in range(10):
    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(.005, .3)
    trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 2)
    trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
    while (np.any(trueParameters) == False):
        trueParameters = np.random.uniform(low=lowerBounds[3], high=upperBounds[3], size= 2)
    
    
    
    
    process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                              volatility = trueVolatility,
                              intensity = np.sum(trueParameters),
                              jumps = trueJumps,
                              jumpProbabilities = trueParameters/np.sum(trueParameters),
                              initialPrice = initialPrice)
    
    
    guessData = process[:,: 390*60 + 1]
    
    optimizationData = process[:,390*60::390]
    optionPrice = ss.jumpDiffusionCallPrice(optimizationData, strikePrices, trueVolatility, trueJumps, trueParameters, error)
    
    volatility = np.mean(np.sqrt(np.sum(sampleBiPowerVariation(guessData, 390)/(60/252), axis = 1)))
    
    intensity =  len(jumps(guessData,390))/(10*60/252)
    
    sizes, probs = jumpCluster(guessData, 390, 2)
    
    guess = np.array([
        volatility,
        sizes[0],
        sizes[1],
        probs[0] * intensity,
        probs[1] * intensity
    ])

    for j in range(len(guess)):
        if guess[j] > upperBounds[j]:
            guess[j] = upperBounds[j] - error
        if guess[j] < lowerBounds[j]:
            guess[j] = lowerBounds[j] + error

    
    start = time.perf_counter()
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, optimizationData, optionPrice), max_nfev=80)
    
    end = time.perf_counter()

    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The inital guesses are {guess}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The estimated jump sizes are {result['x'][1:3]} and the true jump sizes are {trueJumps}")
    print(f"The estimated parameters are {result['x'][3:]} and the true parameters are {trueParameters}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print(f'Time for least squares optimization: {end - start} seconds') 
    print()

Results for Trial 1

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [0.33960515 0.4120075  0.73662723 0.84       7.14      ]
The estimated volatility is 0.325655787774015 and the true volatility is 0.3256557877740467
The estimated jump sizes are [0.41138301 0.73617849] and the true jump sizes are [0.41138301 0.73617849]
The estimated parameters are [1.42841473 9.13382697] and the true parameters are [1.42841473 9.13382697]
The value of the cost function at the estimated parameter is 1.3474292304560475e-24
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 43.16605739999795 seconds

Results for Trial 2

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [ 0.65136866 -0.21719073  0.3546284   5.04        4.62      ]
The estimated volatility is 0.642256275925208 and the true volatility is 0.6422562759252026
The estimated jump sizes are [-0.21684641  0.3541

There is a pretty big improvement! Let's do the same for the case when $M = 3$.

In [58]:
lowerBounds = np.array([
    .05,  # lower bound for volatility
    -0.6, # lower bound for jump size
    -0.6,
    -0.6,
    0,    # lower bound for parameters
    0,
    0
])

upperBounds = np.array([
    .9,  # upper bound for volatility
    1,   # upper bound for jump size
    1,
    1,
    10,  # upper bound for parameters
    10,
    10
])

for i in range(10):
    
    trueVolatility = random.uniform(lowerBounds[0], upperBounds[0])
    trueMeanRateOfReturn = random.uniform(.005, .3)
    trueJumps = np.random.uniform(low=lowerBounds[1], high=upperBounds[1], size= 3)
    trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
    while (np.any(trueParameters) == False):
        trueParameters = np.random.uniform(low=lowerBounds[4], high=upperBounds[4], size= 3)
    
    
    
    
    process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                              volatility = trueVolatility,
                              intensity = np.sum(trueParameters),
                              jumps = trueJumps,
                              jumpProbabilities = trueParameters/np.sum(trueParameters),
                              initialPrice = initialPrice)
    
    guessData = process[:,: 390*60 + 1]
    
    optimizationData = process[:,390*60::390]
    
    optionPrice = ss.jumpDiffusionCallPrice(optimizationData, strikePrices, trueVolatility, trueJumps, trueParameters, error)
    
    volatility = np.mean(np.sqrt(np.sum(sampleBiPowerVariation(guessData, 390)/(60/252), axis = 1)))
    
    intensity =  len(jumps(guessData,390))/(10*60/252)
    
    sizes, probs = jumpCluster(guessData, 390, 3)
    
    guess = np.array([
        volatility,
        sizes[0],
        sizes[1],
        sizes[2],
        probs[0] * intensity,
        probs[1] * intensity,
        probs[2] * intensity
    ])
    
    for j in range(len(guess)):
        if guess[j] > upperBounds[j]:
            guess[j] = upperBounds[j] - error
        if guess[j] < lowerBounds[j]:
            guess[j] = lowerBounds[j] + error
    
    
    
    start = time.perf_counter()
    
    result = least_squares(error_prices, x0 = guess, bounds = (lowerBounds, upperBounds), args = (strikePrices, optimizationData, optionPrice), max_nfev=80)
    
    end = time.perf_counter()
    
    print(f"Results for Trial {i+1}")
    print()
    print(f'Message: ' + result['message'])
    print(f"Success status: {result['success']}")
    print(f"The inital guesses are {guess}")
    print(f"The estimated volatility is {result['x'][0]} and the true volatility is {trueVolatility}")
    print(f"The estimated jump sizes are {result['x'][1:4]} and the true jump sizes are {trueJumps}")
    print(f"The estimated parameters are {result['x'][4:]} and the true parameters are {trueParameters}")
    print(f"The value of the cost function at the estimated parameter is {result['cost']}")
    print(f"Are there any parameters that are at the bounds?: {np.any(result['active_mask'])}")
    print(f'Time for least squares optimization: {end - start} seconds') 
    print()

Results for Trial 1

Message: `gtol` termination condition is satisfied.
Success status: True
The inital guesses are [ 0.1710959  -0.41175209  0.69958097  0.86904936  5.88        9.999999
  4.62      ]
The estimated volatility is 0.12886414232997168 and the true volatility is 0.1288641423299192
The estimated jump sizes are [-0.41183027  0.69960138  0.86897914] and the true jump sizes are [ 0.69960138 -0.41183027  0.86897914]
The estimated parameters are [5.17898108 9.88567579 5.13639281] and the true parameters are [9.88567579 5.17898108 5.13639281]
The value of the cost function at the estimated parameter is 3.07486561663925e-24
Are there any parameters that are at the bounds?: False
Time for least squares optimization: 224.93595110002207 seconds

Results for Trial 2

Message: `ftol` termination condition is satisfied.
Success status: True
The inital guesses are [0.54286656 0.65185796 0.6564854  0.792737   4.62       3.78
 1.26      ]
The estimated volatility is 0.5941315993190449 and

Nonlinear least squares converges at a higher rate than before!